# Upload zipped directory and unzip it in Azure blob storage

In [ ]:
import os
import zipfile
from azure.storage.blob import BlobServiceClient

In [ ]:
# Configuration
connect_str = ""  # Replace with your Azure Storage connection string
container_name = ""  # Replace with your container name
blob_prefix = "datasets/segmentation_data_zip/"  # Path to the zip files in blob storage
local_download_path = "./temp"  # Local directory for temporary storage
local_extract_path = "./temp/unzipped"  # Local directory for extracted files

In [ ]:
# Create blob service client
blob_service_client = BlobServiceClient.from_connection_string(connect_str)
container_client = blob_service_client.get_container_client(container_name)

In [ ]:
# Ensure local directories exist
os.makedirs(local_download_path, exist_ok=True)
os.makedirs(local_extract_path, exist_ok=True)

In [ ]:
# List blobs in the target directory
blob_list = container_client.list_blobs(name_starts_with=blob_prefix)

In [ ]:
for blob in blob_list:
    # Download each zip file
    blob_client = container_client.get_blob_client(blob)
    local_zip_path = os.path.join(local_download_path, os.path.basename(blob.name))
    print(f"Downloading {blob.name} to {local_zip_path}")
    with open(local_zip_path, "wb") as download_file:
        download_file.write(blob_client.download_blob().readall())

    # Unzip the file
    print(f"Unzipping {local_zip_path}")
    with zipfile.ZipFile(local_zip_path, 'r') as zip_ref:
        zip_ref.extractall(local_extract_path)

    # Upload extracted files back to the blob storage
    for root, _, files in os.walk(local_extract_path):
        for file in files:
            local_file_path = os.path.join(root, file)
            blob_path = os.path.join(blob_prefix, "unzipped", os.path.relpath(local_file_path, local_extract_path))
            print(f"Uploading {local_file_path} to {blob_path}")
            with open(local_file_path, "rb") as data:
                container_client.upload_blob(name=blob_path, data=data, overwrite=True)

    # Clean up local files
    os.remove(local_zip_path)
    for root, _, files in os.walk(local_extract_path):
        for file in files:
            os.remove(os.path.join(root, file))

print("All files unzipped and uploaded successfully.")

# Register Datasets

In [ ]:
f2c67079-16e2-4ab7-82ee-0c438d92b95e

In [1]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

# Authenticate and create an MLClient instance
credential = DefaultAzureCredential()
ml_client = MLClient.from_config(credential)

Found the config file in: /config.json


In [3]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

# Define dataset paths
dataset_name = "segmentation-train-images"
description = "Dataset for segmentation training (images)"
data_path = 'azureml://subscriptions/f2c67079-16e2-4ab7-82ee-0c438d92b95e/resourcegroups/oc-central/workspaces/oc-central/datastores/workspaceblobstore/paths/datasets/segmentation_data_zip/unzipped/leftImg8bit/train/'
# data_path = "azureml://datastores/workspaceblobstore/paths/segmentation-dataset/train/images"

# Register the dataset
segmentation_dataset = Data(
    name=dataset_name,
    description=description,
    path=data_path,
    type=AssetTypes.URI_FOLDER,  # URI_FOLDER for folders
    tags={"type": "segmentation training images", "task": "image-segmentation"}
)

ml_client.data.create_or_update(segmentation_dataset)
print(f"Dataset '{dataset_name}' registered successfully.")


Dataset 'segmentation-train-images' registered successfully.


In [4]:
# from azure.ai.ml.entities import Data
# from azure.ai.ml.constants import AssetTypes

# Define dataset paths
dataset_name = "segmentation-train-labels-masks"
description = "Dataset for segmentation training (training labels/masks)"
data_path = 'azureml://subscriptions/f2c67079-16e2-4ab7-82ee-0c438d92b95e/resourcegroups/oc-central/workspaces/oc-central/datastores/workspaceblobstore/paths/datasets/segmentation_data_zip/unzipped/gtFine/train/'
# data_path = "azureml://datastores/workspaceblobstore/paths/segmentation-dataset/train/images"

# Register the dataset
segmentation_dataset = Data(
    name=dataset_name,
    description=description,
    path=data_path,
    type=AssetTypes.URI_FOLDER,  # URI_FOLDER for folders
    tags={"type": "segmentation training masks", "task": "image-segmentation"}
)

ml_client.data.create_or_update(segmentation_dataset)
print(f"Dataset '{dataset_name}' registered successfully.")

Dataset 'segmentation-train-labels-masks' registered successfully.


In [5]:
# Define dataset paths
dataset_name = "segmentation-validation-images"
description = "Dataset for segmentation validation (images)"
data_path = 'azureml://subscriptions/f2c67079-16e2-4ab7-82ee-0c438d92b95e/resourcegroups/oc-central/workspaces/oc-central/datastores/workspaceblobstore/paths/datasets/segmentation_data_zip/unzipped/leftImg8bit/val/'
# data_path = "azureml://datastores/workspaceblobstore/paths/segmentation-dataset/train/images"

# Register the dataset
segmentation_dataset = Data(
    name=dataset_name,
    description=description,
    path=data_path,
    type=AssetTypes.URI_FOLDER,  # URI_FOLDER for folders
    tags={"type": "segmentation validation images", "task": "image-segmentation"}
)

ml_client.data.create_or_update(segmentation_dataset)
print(f"Dataset '{dataset_name}' registered successfully.")

Dataset 'segmentation-validation-images' registered successfully.


In [6]:
# Define dataset paths
dataset_name = "segmentation-validation-labels-masks"
description = "Dataset for segmentation validation (validation labels/masks)"
data_path = 'azureml://subscriptions/f2c67079-16e2-4ab7-82ee-0c438d92b95e/resourcegroups/oc-central/workspaces/oc-central/datastores/workspaceblobstore/paths/datasets/segmentation_data_zip/unzipped/gtFine/val/'
# data_path = "azureml://datastores/workspaceblobstore/paths/segmentation-dataset/train/images"

# Register the dataset
segmentation_dataset = Data(
    name=dataset_name,
    description=description,
    path=data_path,
    type=AssetTypes.URI_FOLDER,  # URI_FOLDER for folders
    tags={"type": "segmentation validation masks", "task": "image-segmentation"}
)

ml_client.data.create_or_update(segmentation_dataset)
print(f"Dataset '{dataset_name}' registered successfully.")

Dataset 'segmentation-validation-labels-masks' registered successfully.


In [ ]:
# import os
# from azureml.core import Dataset, Workspace
# import numpy as np

# ws = Workspace.from_config()
# datastore = ws.get_default_datastore()

In [13]:

from azureml.fsspec import AzureMachineLearningFileSystem
fs = AzureMachineLearningFileSystem('azureml://subscriptions/f2c67079-16e2-4ab7-82ee-0c438d92b95e/resourcegroups/oc-central/workspaces/oc-central/datastores/workspaceblobstore/paths/datasets/segmentation_data_zip/unzipped/gtFine/val/')


In [23]:
p = None
for p in fs.glob("*_gtFine_labelIds.png*"):
    print(path)
    break

In [24]:
# Azure Machine Learning workspace details:
subscription = '<subscription_id>'
resource_group = '<resource_group>'
workspace = '<workspace>'
datastore_name = '<datastore>'
path_on_datastore = '<path>'

# long-form Datastore uri format:
uri = f'azureml://subscriptions/{subscription}/resourcegroups/{resource_group}/workspaces/{workspace}/datastores/{datastore_name}/paths/{path_on_datastore}'

In [15]:
fs.ls()

['datasets/segmentation_data_zip/unzipped/gtFine/val/frankfurt/',
 'datasets/segmentation_data_zip/unzipped/gtFine/val/lindau/',
 'datasets/segmentation_data_zip/unzipped/gtFine/val/munster/']

In [ ]:
azureml://subscriptions/f2c67079-16e2-4ab7-82ee-0c438d92b95e/resourcegroups/oc-central/workspaces/oc-central/datastores/workspaceblobstore/paths/datasets/segmentation_data_zip/unzipped/gtFine/val/

In [ ]:
# Register the images dataset
images_train_dataset = Dataset.File.from_files(path=(datastore,'datasets/segmentation_data_zip/unzipped/leftImg8bit/train'))
images_train_dataset.register(workspace=ws,
                                name='segmentation_images_train',
                                description='[Training] Dataset containing input images for segmentation',
                                create_new_version=True)

In [ ]:
# Register the images dataset
images_train_dataset = Dataset.File.from_files(path=(datastore,'datasets/segmentation_data_zip/unzipped/leftImg8bit/val'))
images_train_dataset.register(workspace=ws,
                                name='segmentation_images_val',
                                description='[Validation] Dataset containing input images for segmentation',
                                create_new_version=True)

In [ ]:
# Register the images dataset
images_train_dataset = Dataset.File.from_files(path=(datastore,'datasets/segmentation_data_zip/unzipped/leftImg8bit/test'))
images_train_dataset.register(workspace=ws,
                                name='segmentation_images_test',
                                description='[Testing] Dataset containing input images for testing',
                                create_new_version=True)

In [ ]:
# Register the masks dataset
masks_dataset = Dataset.File.from_files(path=(datastore, 'datasets/segmentation_data_zip/unzipped/gtFine/train'))
masks_dataset.register(workspace=ws,
                       name='segmentation_masks_train',
                       description='[Train] Dataset containing masks for segmentation',
                       create_new_version=True)
# print("Datasets registered successfully!")

In [ ]:
# Register the masks dataset
masks_dataset = Dataset.File.from_files(path=(datastore, 'datasets/segmentation_data_zip/unzipped/gtFine/val'))
masks_dataset.register(workspace=ws,
                       name='segmentation_masks_val',
                       description='[Validation] Dataset containing masks for segmentation',
                       create_new_version=True)
# print("Datasets registered successfully!")

In [ ]:
# Register the masks dataset
masks_dataset = Dataset.File.from_files(path=(datastore, 'datasets/segmentation_data_zip/unzipped/gtFine/test'))
masks_dataset.register(workspace=ws,
                       name='segmentation_masks_test',
                       description='[Testing] Dataset containing masks for segmentation',
                       create_new_version=True)

# Download Datasets

In [5]:
%load_ext autoreload
%autoreload 2

In [7]:
from azureml.fsspec import AzureMachineLearningFileSystem
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

In [ ]:
from azureml.fsspec import AzureMachineLearningFileSystem
# instantiate file system using following URI
fs = AzureMachineLearningFileSystem('azureml://subscriptions/<subid>/resourcegroups/<rgname>/workspaces/<workspace_name>/datastores/<datastorename>/paths/')

# you can specify recursive as False to upload a file
# fs.upload(lpath='data/upload_files/crime-spring.csv', rpath='data/fsspec', recursive=False, **{'overwrite': 'MERGE_WITH_OVERWRITE'})

# you need to specify recursive as True to upload a folder
fs.upload(lpath='data/upload_folder/', rpath='data/fsspec_folder', recursive=True, **{'overwrite': 'MERGE_WITH_OVERWRITE'})

In [2]:
# from azure.ai.ml import MLClient
# from azure.identity import DefaultAzureCredential

# # authenticate
# credential = DefaultAzureCredential()

SUBSCRIPTION = "f2c67079-16e2-4ab7-82ee-0c438d92b95e"
RESOURCE_GROUP = "oc-central"
WS_NAME = "oc-central"
DATASTORE = "workspaceblobstore"
# Get a handle to the workspace
# ml_client = MLClient(
#     credential=credential,
#     subscription_id=SUBSCRIPTION,
#     resource_group_name=RESOURCE_GROUP,
#     workspace_name=WS_NAME,
# )

In [71]:
# long-form Datastore uri format:
uri = f'azureml://subscriptions/{SUBSCRIPTION}/resourcegroups/{RESOURCE_GROUP}/workspaces/{WS_NAME}/datastores/{DATASTORE}/paths/'

In [72]:
fs = AzureMachineLearningFileSystem(uri) #'azureml://subscriptions/f2c67079-16e2-4ab7-82ee-0c438d92b95e/resourcegroups/oc-central/workspaces/oc-central/datastores/workspaceblobstore/paths/datasets/segmentation_data_zip/unzipped/gtFine/val/')

In [73]:
fs.ls()

['datasets/']

In [8]:
credential = DefaultAzureCredential()
ml_client = MLClient.from_config(credential)

Found the config file in: /config.json


In [67]:
# List of dataset names
dataset_paths = [
    "segmentation-train-images",
    "segmentation-train-labels-masks",
     "segmentation-validation-images",
    "segmentation-validation-labels-masks",]
# Mount each dataset

In [10]:
registered_train_images = ml_client.data.get(name=dataset_paths[0], version=1)
training_images = fs.glob(f"{registered_train_images.path}/**_leftImg8bit.png")
print(len(training_images))
training_images[:5]

2975


['datasets/segmentation_data_zip/unzipped/leftImg8bit/train/aachen/aachen_000000_000019_leftImg8bit.png',
 'datasets/segmentation_data_zip/unzipped/leftImg8bit/train/aachen/aachen_000001_000019_leftImg8bit.png',
 'datasets/segmentation_data_zip/unzipped/leftImg8bit/train/aachen/aachen_000002_000019_leftImg8bit.png',
 'datasets/segmentation_data_zip/unzipped/leftImg8bit/train/aachen/aachen_000003_000019_leftImg8bit.png',
 'datasets/segmentation_data_zip/unzipped/leftImg8bit/train/aachen/aachen_000004_000019_leftImg8bit.png']

In [118]:
test_images_path = '/mnt/batch/tasks/shared/LS_root/mounts/clusters/compute-otpimized/code/Users/george.gvishiani/img_segmentation/data/processed/test/images'
test_masks_path = '/mnt/batch/tasks/shared/LS_root/mounts/clusters/compute-otpimized/code/Users/george.gvishiani/img_segmentation/data/processed/test/masks'
train_images_path = '/mnt/batch/tasks/shared/LS_root/mounts/clusters/compute-otpimized/code/Users/george.gvishiani/img_segmentation/data/processed/train/images'
train_masks_path = '/mnt/batch/tasks/shared/LS_root/mounts/clusters/compute-otpimized/code/Users/george.gvishiani/img_segmentation/data/processed/train/masks'
valid_images_path='/mnt/batch/tasks/shared/LS_root/mounts/clusters/compute-otpimized/code/Users/george.gvishiani/img_segmentation/data/p7rocessed/valid/images'
valid_masks_path= '/mnt/batch/tasks/shared/LS_root/mounts/clusters/compute-otpimized/code/Users/george.gvishiani/img_segmentation/data/processed/valid/masks'

In [ ]:
fs.download(rpath=registered_train_images.path, lpath=train_images_path, recursive=True)
fs.download(rpath=registered_train_masks.path, lpath=train_masks_path, recursive=True)
fs.download(rpath=registered_valid_images.path, lpath=valid_images_path, recursive=True)
fs.download(rpath=registered_valid_masks.path, lpath=valid_masks_path, recursive=True)

# Loading dataset

In [1]:
# pip install azure-ai-ml
# pip install azure-identity
# libraries required
# pip install tensorflow, keras
# pip install mlflow 
# pip install azureml-mlflow
# pip install matplotlib

In [58]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [59]:
from data_processing.image_processing import load_dataset_unet, check_paths

In [60]:
dtr, dval = load_dataset_unet()

Cause: could not parse the source code of <function load_dataset_unet.<locals>.<lambda> at 0x7f5f00c2ca60>: no matching AST found among candidates:
# coding=utf-8
lambda image, mask: (read_image(image), read_label(mask))
# coding=utf-8
lambda image, mask: (read_image(image), read_label(mask))
# coding=utf-8
lambda image, mask: (image, map_labels_tf(mask, original_classes, class_mapping, new_labels))
# coding=utf-8
lambda image, mask: (image, map_labels_tf(mask, original_classes, class_mapping, new_labels))
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Cause: could not parse the source code of <function load_dataset_unet.<locals>.<lambda> at 0x7f5f00c2ca60>: no matching AST found among candidates:
# coding=utf-8
lambda image, mask: (read_image(image), read_label(mask))
# coding=utf-8
lambda image, mask: (read_image(image), read_label(mask))
# coding=utf-8
lambda image, mask: (image, map_labels_tf(mask, original_classes, class_mapping, new_

In [150]:
def load_paths(directory):
    full_paths=[]
    for root, dirs, files in os.walk(directory):
        for file in files:
            full_path = os.path.abspath(os.path.join(root, file))
            full_paths.append(full_path)
    return full_paths

In [151]:
train_images = 'data/processed/train/images'
train_images = list_all_files_with_full_paths(train_images)
print(len(train_images))
train_images[:3]

2975


['/mnt/batch/tasks/shared/LS_root/mounts/clusters/compute-otpimized/code/Users/george.gvishiani/img_segmentation/data/processed/train/images/aachen/aachen_000000_000019_leftImg8bit.png',
 '/mnt/batch/tasks/shared/LS_root/mounts/clusters/compute-otpimized/code/Users/george.gvishiani/img_segmentation/data/processed/train/images/aachen/aachen_000001_000019_leftImg8bit.png',
 '/mnt/batch/tasks/shared/LS_root/mounts/clusters/compute-otpimized/code/Users/george.gvishiani/img_segmentation/data/processed/train/images/aachen/aachen_000002_000019_leftImg8bit.png']

In [152]:
train_masks = 'data/processed/train/masks'
train_masks = list_all_files_with_full_paths(train_masks)
train_masks = [msk for msk in train_masks if "_labelIds.png" in msk]
print(len(train_masks))
train_masks[:3]

2975


['/mnt/batch/tasks/shared/LS_root/mounts/clusters/compute-otpimized/code/Users/george.gvishiani/img_segmentation/data/processed/train/masks/aachen/aachen_000000_000019_gtFine_labelIds.png',
 '/mnt/batch/tasks/shared/LS_root/mounts/clusters/compute-otpimized/code/Users/george.gvishiani/img_segmentation/data/processed/train/masks/aachen/aachen_000001_000019_gtFine_labelIds.png',
 '/mnt/batch/tasks/shared/LS_root/mounts/clusters/compute-otpimized/code/Users/george.gvishiani/img_segmentation/data/processed/train/masks/aachen/aachen_000002_000019_gtFine_labelIds.png']

In [153]:
valid_images = 'data/processed/valid/images'
valid_images = list_all_files_with_full_paths(valid_images)
print(len(valid_images))
valid_images[:3]

500


['/mnt/batch/tasks/shared/LS_root/mounts/clusters/compute-otpimized/code/Users/george.gvishiani/img_segmentation/data/processed/valid/images/frankfurt/frankfurt_000000_000294_leftImg8bit.png',
 '/mnt/batch/tasks/shared/LS_root/mounts/clusters/compute-otpimized/code/Users/george.gvishiani/img_segmentation/data/processed/valid/images/frankfurt/frankfurt_000000_000576_leftImg8bit.png',
 '/mnt/batch/tasks/shared/LS_root/mounts/clusters/compute-otpimized/code/Users/george.gvishiani/img_segmentation/data/processed/valid/images/frankfurt/frankfurt_000000_001016_leftImg8bit.png']

In [154]:
valid_masks = 'data/processed/valid/masks'
valid_masks = list_all_files_with_full_paths(valid_masks)
valid_masks = [msk for msk in valid_masks if "_labelIds.png" in msk]
print(len(valid_masks))
valid_masks[:3]

500


['/mnt/batch/tasks/shared/LS_root/mounts/clusters/compute-otpimized/code/Users/george.gvishiani/img_segmentation/data/processed/valid/masks/frankfurt/frankfurt_000000_000294_gtFine_labelIds.png',
 '/mnt/batch/tasks/shared/LS_root/mounts/clusters/compute-otpimized/code/Users/george.gvishiani/img_segmentation/data/processed/valid/masks/frankfurt/frankfurt_000000_000576_gtFine_labelIds.png',
 '/mnt/batch/tasks/shared/LS_root/mounts/clusters/compute-otpimized/code/Users/george.gvishiani/img_segmentation/data/processed/valid/masks/frankfurt/frankfurt_000000_001016_gtFine_labelIds.png']

In [155]:
# cd ../valid

In [158]:
check_paths(train_images, train_masks)

'Paths are correct - check passed'

In [159]:
check_paths(valid_images, valid_masks)

'Paths are correct - check passed'

In [163]:
cwd = Path.cwd()

In [166]:
from pathlib import Path

def find_project_root(current_path, marker_file="README.md"):
    current_path = Path(current_path).resolve()
    while current_path != current_path.root:
        if (current_path / marker_file).exists():
            return current_path
        current_path = current_path.parent
    raise FileNotFoundError(f"Root directory containing {marker_file} not found.")

# Example usage
project_root = find_project_root(cwd)  # Or pass the current directory as Path.cwd()
print(f"Project root: {project_root}")

Project root: /mnt/batch/tasks/shared/LS_root/mounts/clusters/compute-otpimized/code/Users/george.gvishiani/img_segmentation


In [171]:
project_root /'data/processed/train/images'

PosixPath('/mnt/batch/tasks/shared/LS_root/mounts/clusters/compute-otpimized/code/Users/george.gvishiani/img_segmentation/data/processed/train/images')

In [67]:
ml_client.data.get(name=dataset_paths[1], version=1).path

'azureml://subscriptions/f2c67079-16e2-4ab7-82ee-0c438d92b95e/resourcegroups/oc-central/workspaces/oc-central/datastores/workspaceblobstore/paths/datasets/segmentation_data_zip/unzipped/gtFine/train/'

In [109]:
registered_train_masks =  ml_client.data.get(name=dataset_paths[1], version=1)
training_masks = fs.glob(f"{registered_train_masks.path}/**_labelIds.png")
print(len(training_masks))
training_masks[:5]

2975


['datasets/segmentation_data_zip/unzipped/gtFine/train/aachen/aachen_000000_000019_gtFine_labelIds.png',
 'datasets/segmentation_data_zip/unzipped/gtFine/train/aachen/aachen_000001_000019_gtFine_labelIds.png',
 'datasets/segmentation_data_zip/unzipped/gtFine/train/aachen/aachen_000002_000019_gtFine_labelIds.png',
 'datasets/segmentation_data_zip/unzipped/gtFine/train/aachen/aachen_000003_000019_gtFine_labelIds.png',
 'datasets/segmentation_data_zip/unzipped/gtFine/train/aachen/aachen_000004_000019_gtFine_labelIds.png']

In [110]:
registered_valid_images =  ml_client.data.get(name=dataset_paths[2], version=1)
valid_images = fs.glob(f"{registered_valid_images.path}/**_leftImg8bit.png")
print(len(valid_images))
valid_images[:5]

500


['datasets/segmentation_data_zip/unzipped/leftImg8bit/val/frankfurt/frankfurt_000000_000294_leftImg8bit.png',
 'datasets/segmentation_data_zip/unzipped/leftImg8bit/val/frankfurt/frankfurt_000000_000576_leftImg8bit.png',
 'datasets/segmentation_data_zip/unzipped/leftImg8bit/val/frankfurt/frankfurt_000000_001016_leftImg8bit.png',
 'datasets/segmentation_data_zip/unzipped/leftImg8bit/val/frankfurt/frankfurt_000000_001236_leftImg8bit.png',
 'datasets/segmentation_data_zip/unzipped/leftImg8bit/val/frankfurt/frankfurt_000000_001751_leftImg8bit.png']

In [111]:
registered_valid_masks =  ml_client.data.get(name=dataset_paths[3], version=1)
valid_masks = fs.glob(f"{registered_valid_masks.path}/**_labelIds.png")
print(len(valid_masks))
valid_masks[:5]

500


['datasets/segmentation_data_zip/unzipped/gtFine/val/frankfurt/frankfurt_000000_000294_gtFine_labelIds.png',
 'datasets/segmentation_data_zip/unzipped/gtFine/val/frankfurt/frankfurt_000000_000576_gtFine_labelIds.png',
 'datasets/segmentation_data_zip/unzipped/gtFine/val/frankfurt/frankfurt_000000_001016_gtFine_labelIds.png',
 'datasets/segmentation_data_zip/unzipped/gtFine/val/frankfurt/frankfurt_000000_001236_gtFine_labelIds.png',
 'datasets/segmentation_data_zip/unzipped/gtFine/val/frankfurt/frankfurt_000000_001751_gtFine_labelIds.png']

In [59]:
check_paths(training_images, training_masks)

'Paths are correct - check passed'

In [63]:
check_paths(valid_masks, valid_images)

'Paths are correct - check passed'

In [104]:
# for img, lbl in zip(training_images, training_masks):
#     print(read_image(img).shape) 
#     print(read_label(lbl).shape)
#     break

In [68]:
print("Dataset Name:", registered_train_images.name)
print("Dataset Path:", registered_train_images._path)
print("Dataset Type:", registered_train_images.type)

Dataset Name: segmentation-train-images
Dataset Path: azureml://subscriptions/f2c67079-16e2-4ab7-82ee-0c438d92b95e/resourcegroups/oc-central/workspaces/oc-central/datastores/workspaceblobstore/paths/datasets/segmentation_data_zip/unzipped/leftImg8bit/train/
Dataset Type: uri_folder


In [70]:
train_img_path = registered_train_images.path

In [ ]:
# Create an fsspec filesystem
import azureml-fsspec
fs = fsspec.filesystem("azureml", asset=train_img_path)

In [71]:
from azureml.fsspec import AzureMachineLearningFileSystem

In [72]:
fs = AzureMachineLearningFileSystem(uri=train_img_path)

In [75]:
type(files)

list

In [105]:
# # List all files in the dataset
# files = fs.glob("****")  # List all files recursively
# print("Files in dataset:")
# counter = 0
# # for file in files:
# #     print(file)
# len(files)    

In [78]:
files[:5]

['datasets/segmentation_data_zip/unzipped/leftImg8bit/train/aachen/',
 'datasets/segmentation_data_zip/unzipped/leftImg8bit/train/aachen/aachen_000000_000019_leftImg8bit.png',
 'datasets/segmentation_data_zip/unzipped/leftImg8bit/train/aachen/aachen_000001_000019_leftImg8bit.png',
 'datasets/segmentation_data_zip/unzipped/leftImg8bit/train/aachen/aachen_000002_000019_leftImg8bit.png',
 'datasets/segmentation_data_zip/unzipped/leftImg8bit/train/aachen/aachen_000003_000019_leftImg8bit.png']

In [81]:
# Access a specific file
target_file = "datasets/segmentation_data_zip/unzipped/leftImg8bit/train/aachen/aachen_000000_000019_leftImg8bit.png"
if target_file in files:
    with fs.open(target_file, "rb") as f:
        content = f.read()
        print(f"Successfully read file: {target_file}")
else:
    print(f"File {target_file} not found in dataset.")

Successfully read file: datasets/segmentation_data_zip/unzipped/leftImg8bit/train/aachen/aachen_000000_000019_leftImg8bit.png


In [83]:
import tensorflow as tf

In [84]:
# Check if the file exists
if target_file in fs.glob("**"):
    # Read file content as a local temporary file
    local_file_path = f"/tmp/{target_file.split('/')[-1]}"
    with fs.open(target_file, "rb") as src_file, open(local_file_path, "wb") as local_file:
        local_file.write(src_file.read())
    
    # Use tf.io.read_file on the local file
    content = tf.io.read_file(local_file_path)
    print("File content successfully read by TensorFlow!")
else:
    print(f"File {target_file} not found in dataset.")

File content successfully read by TensorFlow!


In [ ]:
pip freeze > requirements_data_processing.txt

# Training Prep (Metrics, Cost Function, Optimization Algorithm, Callbacks \[early stopping, plotting, learning rate decay, Saving best five models / checkpoints, custom history class, tensorboard, mlflow])

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from callbacks.callbacks import keras_callbacks, PlotResultsCallback

from custom_metrics.custom_metrics import iou, dice_coefficient
from model_definitions.unet import unet_with_vgg16_encoder

from utils.utils import keras_optimizer

In [ ]:
# Define my Azure details
# subscription_id = "your_subscription_id"
# resource_group_name = "your_resource_group_name"
# workspace_name = "your_workspace_name"

# Get the Azure MLClient instance
# ml_client, ws = get_ml_client(ws.subscription_id, ws.resource_group, ws.name)

# Log or interact with Azure ML
# ml_client.jobs.upload(path="outputs/plots", name="epoch_plots")

In [ ]:
# from utils.azure_ml_utils import get_ml_client
# import keras
import keras
keras.backend.clear_session()

In [ ]:
early_stop, model_checkpoint, reduce_lr, custom_history, top_k_checkpoint = keras_callbacks()

In [ ]:
plot_callback=PlotResultsCallback(validation_data=dataset_val)

In [ ]:
# Combine custom and builtin metrics for Keras model.fit method

callbacks = [early_stop, model_checkpoint, reduce_lr, plot_callback, custom_history, top_k_checkpoint]
metrics = [dice_coefficient, iou, 'accuracy']

In [ ]:
input_shape = (1024, 2048, 3)  # Input shape for Cityscapes dataset
num_classes = 8  # Number of major groups/classes in mask
model = unet_with_vgg16_encoder(input_shape, num_classes)

In [ ]:
optimizer = keras_optimizer()
model.compile(optimizer=optimizer,
              loss='sparse_categorical_crossentropy',
              metrics=metrics)

In [ ]:
model.summary()

# Training

In [ ]:
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential
from azure.ai.ml import MLClient

try:
    credential = DefaultAzureCredential()
    # Check if given credential can get token successfully.
    credential.get_token("https://management.azure.com/.default")
except Exception as ex:
    # Fall back to InteractiveBrowserCredential in case DefaultAzureCredential not work
    credential = InteractiveBrowserCredential()

In [ ]:
# Get a handle to workspace
ml_client = MLClient.from_config(credential=credential)

In [ ]:
import mlflow
# import mlflow.azureml
from azureml.core import Workspace, Experiment
import mlflow.keras

In [ ]:
# Connect to your Azure ML workspace
ws = Workspace.from_config()

# Set the MLflow tracking URI to point to your Azure ML workspace
mlflow.set_tracking_uri(ws.get_mlflow_tracking_uri())

# Create an experiment in Azure ML
experiment_name = "unet_experiment_v2"
mlflow.set_experiment(experiment_name)

In [ ]:
# keras.config.disable_traceback_filtering()

In [ ]:
import os
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'

In [ ]:
# mlflow.keras.autolog(disable=True)
# Enable autologging


In [ ]:
keras.__version__

In [ ]:
# batch size 6 with environment set to 1

In [ ]:
with mlflow.start_run():
    mlflow.keras.autolog()
    
    history = model.fit(
                    dataset_train.take(1),
                    validation_data=dataset_val(1),
                    epochs=10,
                    callbacks=callbacks)

In [ ]:
mlflow.end_run()

In [ ]:
history = custom_history.history

In [ ]:
# Convert NumPy arrays to lists for JSON serialization
import numpy as np

def convert_history(history):
    converted_history = {}
    for key, values in history.items():
        converted_values = []
        for value in values:
            if isinstance(value, (np.ndarray, np.number)):
                converted_values.append(value.tolist())
            else:
                converted_values.append(value)
        converted_history[key] = converted_values
    return converted_history

converted_history = convert_history(history)


In [ ]:
import json

with open('./outputs/training_history.json', 'w') as f:
    json.dump(converted_history, f)

In [ ]:
for images, masks in dataset_train.take(1):
    print("Images shape:", images.shape)
    print("Masks shape:", masks.shape)

# Deployment

# Monitoring